# Transformer Encoding and Decoding Methods: Complete Guide

This notebook provides a comprehensive explanation of all transformer encoding and decoding methods, their advantages, disadvantages, and optimal use cases.

**Table of Contents:**
1. Positional Encoding Methods
2. Input Encoding Methods
3. Attention Mechanisms (Self-Attention, Multi-Head Attention)
4. Transformer Decoder Architecture
5. Decoding Strategies
6. Practical Implementations
7. Comparison & Best Practices

## Part 1: Positional Encoding Methods

Since transformers don't have inherent positional information (unlike RNNs), we need to inject position information into the embeddings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from typing import Tuple

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

### 1.1 Absolute Positional Encoding (Sinusoidal - Original Transformer)

**Formula:**
- PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
- PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

**Process:**
- For each position in the sequence
- For each dimension in the embedding
- Use sine/cosine waves with different frequencies

**Advantages:**
- No learnable parameters
- Computational efficient
- Can handle sequences longer than training data
- Continuous representation

**Disadvantages:**
- Fixed and doesn't adapt to data
- Sensitive to absolute positions

**Best for:** General-purpose transformers, long sequences

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Original sinusoidal positional encoding from 'Attention is All You Need'"""
    
    def __init__(self, d_model: int, max_seq_len: int = 5000):
        super().__init__()
        self.d_model = d_model
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the angles
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             -(np.log(10000.0) / d_model))
        
        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))  # Add batch dimension
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        Returns:
            x + positional encoding
        """
        return x + self.pe[:, :x.size(1), :]

# Example usage
print("=" * 70)
print("SINUSOIDAL POSITIONAL ENCODING")
print("=" * 70)

d_model = 64
seq_len = 100
pe_layer = SinusoidalPositionalEncoding(d_model=d_model, max_seq_len=seq_len)

# Visualize positional encoding
plt.figure(figsize=(12, 4))
plt.imshow(pe_layer.pe[0, :50, :].detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='PE Value')
plt.title('Sinusoidal Positional Encoding (first 50 positions)')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position')
plt.tight_layout()
plt.show()

print(f"Shape of PE matrix: {pe_layer.pe.shape}")
print(f"Example PE for position 0: {pe_layer.pe[0, 0, :8]}")
print(f"Example PE for position 10: {pe_layer.pe[0, 10, :8]}")

### 1.2 Learned Positional Encoding (Absolute)

**Process:**
- Create a learnable embedding table for positions
- Each position has a learnable embedding vector
- Trained jointly with the model

**Advantages:**
- Adapts to the specific task
- Can learn meaningful positional patterns
- Often performs slightly better in practice

**Disadvantages:**
- Limited to sequence length seen during training
- More parameters
- Cannot generalize to longer sequences

**Best for:** Fixed-length sequences, task-specific tuning

In [ ]:
class LearnedPositionalEncoding(nn.Module):
    """Learned positional encoding"""
    
    def __init__(self, d_model: int, max_seq_len: int = 512):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # Learnable positional embeddings
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        Returns:
            x + learned positional encoding
        """
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, dtype=torch.long, device=x.device)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)  # (1, seq_len, d_model)
        return x + pos_emb

print("\n" + "=" * 70)
print("LEARNED POSITIONAL ENCODING")
print("=" * 70)

learned_pe = LearnedPositionalEncoding(d_model=d_model, max_seq_len=seq_len)
print(f"Number of parameters: {d_model * seq_len}")
print(f"Learnable embeddings shape: {learned_pe.pos_embedding.weight.shape}")
print(f"\nAdvantage: Task-specific, can learn meaningful patterns")
print(f"Disadvantage: Cannot handle sequences longer than {seq_len}")

### 1.3 Relative Positional Encoding (Shaw et al.)

**Process:**
- Encode relative distances between positions instead of absolute positions
- Applied directly in attention mechanism
- Allows attention to focus on relative relationships

**Advantages:**
- Better for longer sequences
- Can generalize to unseen sequence lengths
- Captures relative distances naturally

**Disadvantages:**
- More complex implementation
- Requires modification of attention mechanism

**Best for:** Long sequences, translation tasks

In [ ]:
class RelativePositionalEncoding(nn.Module):
    """Relative positional encoding (Shaw et al.)"""
    
    def __init__(self, d_model: int, max_relative_distance: int = 128):
        super().__init__()
        self.d_model = d_model
        self.max_relative_distance = max_relative_distance
        
        # Learnable relative position embeddings
        # Size: 2 * max_relative_distance + 1 (for both positive and negative distances)
        self.relative_pos_bias = nn.Embedding(
            2 * max_relative_distance + 1, 
            d_model // 8  # Typically used with multi-head attention
        )
    
    def forward(self, seq_len: int) -> torch.Tensor:
        """
        Generate relative position bias matrix
        Args:
            seq_len: Sequence length
        Returns:
            Relative position bias matrix of shape (seq_len, seq_len)
        """
        # Create position index matrix
        positions = torch.arange(seq_len, dtype=torch.long).unsqueeze(1)
        distances = positions - positions.transpose(0, 1)  # (seq_len, seq_len)
        
        # Clip distances to max_relative_distance
        distances = torch.clamp(
            distances, 
            -self.max_relative_distance, 
            self.max_relative_distance
        )
        
        # Shift to positive indices (add max_relative_distance)
        distances = distances + self.max_relative_distance
        
        # Get embeddings
        relative_bias = self.relative_pos_bias(distances)
        
        return relative_bias

print("\n" + "=" * 70)
print("RELATIVE POSITIONAL ENCODING")
print("=" * 70)

relative_pe = RelativePositionalEncoding(d_model=64, max_relative_distance=128)
seq_len_example = 10
relative_bias = relative_pe(seq_len_example)
print(f"Relative position bias shape: {relative_bias.shape}")
print(f"Captures relative distances, not absolute positions")
print(f"Can handle longer sequences than training data")

### 1.4 Alibi (Attention with Linear Biases)

**Process:**
- Add a linear bias directly to attention scores
- Bias = -|i - j| * slope (slope depends on head)
- No embeddings needed

**Advantages:**
- Extremely simple, no embeddings
- Excellent extrapolation to longer sequences
- Low computational cost

**Disadvantages:**
- Less flexibility than learned positional encoding
- Linear bias might be too restrictive

**Best for:** Long sequences, scaling to very long contexts

In [ ]:
class ALiBi(nn.Module):
    """Attention with Linear Biases (Press et al.)"""
    
    def __init__(self, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        
        # Compute slopes for each head
        # Slopes decrease exponentially: 2^(-8/num_heads), 2^(-16/num_heads), ...
        slopes = torch.Tensor([2 ** (-8 / num_heads * i) for i in range(num_heads)])
        self.register_buffer('slopes', slopes)
    
    def forward(self, seq_len: int) -> torch.Tensor:
        """
        Generate ALiBi bias matrix
        Args:
            seq_len: Sequence length
        Returns:
            Attention bias of shape (num_heads, seq_len, seq_len)
        """
        positions = torch.arange(seq_len, dtype=torch.float).unsqueeze(1)
        distance = positions - positions.transpose(0, 1)  # (seq_len, seq_len)
        
        # Apply slopes for each head
        # distance * slope gives linear bias based on distance
        alibi_bias = distance.abs().unsqueeze(0) * self.slopes.unsqueeze(1).unsqueeze(2)
        alibi_bias = -alibi_bias  # Negative because we penalize distant positions
        
        return alibi_bias

print("\n" + "=" * 70)
print("ALiBi (ATTENTION WITH LINEAR BIASES)")
print("=" * 70)

alibi = ALiBi(num_heads=8)
alibi_bias = alibi(seq_len=20)
print(f"ALiBi bias shape: {alibi_bias.shape}")
print(f"Slopes (per head): {alibi.slopes}")
print(f"\nExample bias for first head (first 5x5):")
print(alibi_bias[0, :5, :5])
print(f"\nProperty: Closer positions have higher attention score")

### 1.5 Rotary Position Embedding (RoPE)

**Process:**
- Rotate query and key vectors by position-dependent angle
- Rotation matrix: 2D rotation applied to consecutive dimension pairs
- θ = base^(-2i/d) where base typically 10000

**Advantages:**
- Excellent extrapolation properties
- Maintains distance information
- Used in modern models (LLaMA, PaLM, etc.)
- Better than ALiBi in many cases

**Disadvantages:**
- More complex implementation
- Requires modifying query/key computation

**Best for:** Large language models, long context

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) - Su et al."""
    
    def __init__(self, d_model: int, base: int = 10000, device: str = 'cpu'):
        super().__init__()
        self.d_model = d_model
        self.base = base
        self.device = device
        
        # Pre-compute inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
    
    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        """
        Apply rotary positional embedding
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
            seq_len: Sequence length
        Returns:
            Position embeddings
        """
        # Compute angles
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.device)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)  # (seq_len, d_model/2)
        
        # Create full frequency matrix (duplicate for both sin and cos)
        emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, d_model)
        
        # Create rotation matrix
        cos_cached = emb.cos().unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, d_model)
        sin_cached = emb.sin().unsqueeze(0).unsqueeze(0)
        
        return cos_cached, sin_cached
    
    @staticmethod
    def rotate_half(x: torch.Tensor) -> torch.Tensor:
        """Rotate half the hidden dims of the input."""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    
    @staticmethod
    def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, 
                             cos: torch.Tensor, sin: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply rotary embeddings to query and key tensors."""
        q_rot = (q * cos) + (RotaryPositionalEmbedding.rotate_half(q) * sin)
        k_rot = (k * cos) + (RotaryPositionalEmbedding.rotate_half(k) * sin)
        return q_rot, k_rot

print("\n" + "=" * 70)
print("ROTARY POSITION EMBEDDING (RoPE)")
print("=" * 70)

rope = RotaryPositionalEmbedding(d_model=64, base=10000)
cos_emb, sin_emb = rope(None, seq_len=100)
print(f"Cosine embedding shape: {cos_emb.shape}")
print(f"Sine embedding shape: {sin_emb.shape}")
print(f"\nUsed by: LLaMA, PaLM, MPT, and other modern LLMs")
print(f"Best feature: Excellent extrapolation to longer sequences")

## Part 2: Input Encoding Methods

These methods encode the input tokens/text into continuous representations.

### 2.1 Token Embedding + Positional Encoding

**Process:**
1. Convert token IDs to embeddings using embedding table
2. Scale embeddings by √d_model
3. Add positional encoding
4. Apply dropout

**Advantages:**
- Simple and effective
- Standard in all transformers
- Separates content and position information

**Disadvantages:**
- Additive combination might lose information
- Fixed vocabulary size

**Best for:** Standard NLP tasks

In [ ]:
class InputEncoding(nn.Module):
    """Standard input encoding: token embedding + positional encoding"""
    
    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional encoding
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: Tensor of shape (batch_size, seq_len)
        Returns:
            Encoded embeddings of shape (batch_size, seq_len, d_model)
        """
        # Token embedding
        x = self.token_embedding(input_ids) * np.sqrt(self.d_model)
        
        # Add positional encoding
        x = x + self.pe[:, :x.size(1), :]
        
        return self.dropout(x)

print("\n" + "=" * 70)
print("INPUT ENCODING (Token Embedding + Positional Encoding)")
print("=" * 70)

vocab_size = 1000
d_model = 512
input_encoder = InputEncoding(vocab_size=vocab_size, d_model=d_model)

# Example
batch_size = 2
seq_len = 10
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
encoded = input_encoder(input_ids)

print(f"Input IDs shape: {input_ids.shape}")
print(f"Encoded output shape: {encoded.shape}")
print(f"\nStep-by-step:")
print(f"1. Token embedding: vocab_size={vocab_size} → d_model={d_model}")
print(f"2. Scale by √d_model = {np.sqrt(d_model):.2f}")
print(f"3. Add positional encoding")
print(f"4. Apply dropout for regularization")

### 2.2 Segment/Token Type Embeddings (for BERT-like models)

**Process:**
- Add an additional embedding for segment IDs
- Useful for tasks with multiple input sequences (e.g., [CLS] Sentence1 [SEP] Sentence2)

**Advantages:**
- Helps model distinguish between different segments
- Essential for sentence pair tasks

**Disadvantages:**
- Extra parameters
- Limited to number of segments defined during training

**Best for:** Multi-segment tasks like BERT

In [ ]:
class BERTInputEncoding(nn.Module):
    """BERT-style input encoding with segment embeddings"""
    
    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 512, 
                 num_segments: int = 2, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional embedding
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        
        # Segment (token type) embedding
        self.segment_embedding = nn.Embedding(num_segments, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_ids: torch.Tensor, segment_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: Tensor of shape (batch_size, seq_len)
            segment_ids: Tensor of shape (batch_size, seq_len)
        Returns:
            Encoded embeddings of shape (batch_size, seq_len, d_model)
        """
        seq_len = input_ids.size(1)
        positions = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        
        # Token + Position + Segment embeddings
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(positions)
        seg_emb = self.segment_embedding(segment_ids)
        
        # Sum all embeddings
        x = token_emb + pos_emb + seg_emb
        
        return self.dropout(x)

print("\n" + "=" * 70)
print("BERT-STYLE INPUT ENCODING")
print("=" * 70)

bert_encoder = BERTInputEncoding(vocab_size=1000, d_model=512, num_segments=2)

input_ids = torch.randint(0, 1000, (2, 10))
segment_ids = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
                             [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]])

encoded = bert_encoder(input_ids, segment_ids)
print(f"Input IDs: {input_ids[0]}")
print(f"Segment IDs: {segment_ids[0]}")
print(f"Output shape: {encoded.shape}")
print(f"\nBenefits:")
print(f"- Model learns to distinguish Sentence A (segment 0) from Sentence B (segment 1)")
print(f"- Essential for tasks like Next Sentence Prediction")

## Part 3: Attention Mechanisms

Attention mechanisms are the core of transformers. Let's understand different variants.

### 3.1 Scaled Dot-Product Attention

**Formula:**
Attention(Q, K, V) = softmax(QK^T / √d_k) V

**Process:**
1. Compute attention scores: QK^T
2. Scale by 1/√d_k
3. Apply softmax to get attention weights
4. Multiply with values
5. Optionally apply attention mask for padding/causality

**Advantages:**
- Efficient matrix operations
- Clear interpretability
- Parallelizable

**Disadvantages:**
- O(n²) complexity
- Can have long-range dependency issues

**Best for:** Most standard tasks

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention"""
    
    def __init__(self, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, 
                mask: torch.Tensor = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            Q: Query tensor of shape (batch, heads, seq_len, d_k)
            K: Key tensor of shape (batch, heads, seq_len, d_k)
            V: Value tensor of shape (batch, heads, seq_len, d_v)
            mask: Mask tensor for padding/causality
        Returns:
            output: Attended values
            attention_weights: Attention probability distribution
        """
        d_k = K.size(-1)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        # Apply mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Apply softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

print("\n" + "=" * 70)
print("SCALED DOT-PRODUCT ATTENTION")
print("=" * 70)

attention = ScaledDotProductAttention(dropout=0.1)

# Example
batch_size, num_heads, seq_len, d_k = 2, 8, 10, 64
Q = torch.randn(batch_size, num_heads, seq_len, d_k)
K = torch.randn(batch_size, num_heads, seq_len, d_k)
V = torch.randn(batch_size, num_heads, seq_len, d_k)

output, attention_weights = attention(Q, K, V)

print(f"Query shape: {Q.shape}")
print(f"Key shape: {K.shape}")
print(f"Value shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights sum to 1 per query:")
print(f"Sum of attention weights: {attention_weights[0, 0, 0, :].sum().item():.4f}")

### 3.2 Multi-Head Attention

**Process:**
1. Project Q, K, V into h different subspaces
2. Apply attention in each subspace
3. Concatenate results
4. Project to output dimension

**Advantages:**
- Attends to different representation subspaces
- Captures multiple types of relationships
- More expressive than single-head attention

**Disadvantages:**
- More parameters
- Slightly higher computational cost

**Best for:** All transformer architectures

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention"""
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(dropout=dropout)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, 
                mask: torch.Tensor = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            Q: Query tensor of shape (batch, seq_len, d_model)
            K: Key tensor of shape (batch, seq_len, d_model)
            V: Value tensor of shape (batch, seq_len, d_model)
            mask: Attention mask
        Returns:
            output: Multi-head attention output
            attention_weights: Attention weights
        """
        batch_size = Q.size(0)
        
        # Project and split into multiple heads
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attended_values, attention_weights = self.attention(Q, K, V, mask)
        
        # Concatenate heads
        output = attended_values.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        
        # Output projection
        output = self.W_o(output)
        output = self.dropout(output)
        
        return output, attention_weights

print("\n" + "=" * 70)
print("MULTI-HEAD ATTENTION")
print("=" * 70)

multi_head = MultiHeadAttention(d_model=512, num_heads=8, dropout=0.1)

# Example
seq_len = 10
X = torch.randn(2, seq_len, 512)
output, attention_weights = multi_head(X, X, X)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nEach head attends to d_k = {512 // 8} dimensions")
print(f"Total: 8 heads × 64 dims = 512 dims")
print(f"\nAdvantage: Each head learns different types of relationships")

### 3.3 Self-Attention vs Cross-Attention

**Self-Attention (Encoder Attention):**
- Q, K, V all come from the same source
- Attends to all positions in the sequence
- Allows position i to attend to position j

**Cross-Attention (Decoder-Encoder Attention):**
- Q comes from decoder
- K, V come from encoder
- Allows decoder to attend to encoder outputs

**Advantages of Self-Attention:**
- Captures internal dependencies
- No information from other sequences needed

**Advantages of Cross-Attention:**
- Bridges encoder and decoder
- Essential for seq2seq models

**Best for:** Self-Attention in encoders; Cross-Attention in decoder-encoder models

In [ ]:
# Self-Attention vs Cross-Attention
print("\n" + "=" * 70)
print("SELF-ATTENTION vs CROSS-ATTENTION")
print("=" * 70)

# Self-Attention Example
print("\n1. SELF-ATTENTION (Encoder)")
print("-" * 50)
encoder_input = torch.randn(2, 10, 512)
self_attention = MultiHeadAttention(d_model=512, num_heads=8)

# Q, K, V all from same source
output_self, _ = self_attention(encoder_input, encoder_input, encoder_input)
print(f"Input shape: {encoder_input.shape}")
print(f"Output shape: {output_self.shape}")
print(f"Q, K, V source: SAME (encoder input)")
print(f"Process: Each position attends to all positions")
print(f"Use case: Encoder layer")

# Cross-Attention Example
print("\n2. CROSS-ATTENTION (Decoder)")
print("-" * 50)
decoder_input = torch.randn(2, 8, 512)
encoder_output = torch.randn(2, 10, 512)
cross_attention = MultiHeadAttention(d_model=512, num_heads=8)

# Q from decoder, K,V from encoder
output_cross, _ = cross_attention(decoder_input, encoder_output, encoder_output)
print(f"Query (decoder) shape: {decoder_input.shape}")
print(f"Key/Value (encoder) shape: {encoder_output.shape}")
print(f"Output shape: {output_cross.shape}")
print(f"Process: Decoder positions attend to encoder positions")
print(f"Use case: Decoder layer (seq2seq)")

### 3.4 Masked Attention (Causal/Autoregressive)

**Process:**
- Apply mask to prevent attention to future positions
- Position i can only attend to positions 0 to i
- Essential for autoregressive decoding

**Advantages:**
- Enables autoregressive generation
- Prevents information leakage from future

**Disadvantages:**
- Limits bidirectional context

**Best for:** Decoder layers, language model pretraining

In [ ]:
def create_causal_mask(seq_len: int, device: str = 'cpu') -> torch.Tensor:
    """
    Create a causal mask for autoregressive attention.
    Position i can only attend to positions 0 to i.
    """
    # Create upper triangular matrix of 1s (future positions)
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    # Invert: 0 for positions to attend to, 1 for positions to mask
    mask = ~mask
    return mask.to(device)

def create_padding_mask(seq_len: int, actual_len: int, device: str = 'cpu') -> torch.Tensor:
    """
    Create a padding mask for variable length sequences.
    """
    mask = torch.ones(seq_len, seq_len).to(device)
    mask[:, actual_len:] = 0
    return mask

print("\n" + "=" * 70)
print("MASKED ATTENTION")
print("=" * 70)

seq_len = 6
causal_mask = create_causal_mask(seq_len)
print(f"Causal Mask ({seq_len}×{seq_len}):")
print(f"1 = can attend, 0 = cannot attend")
print(causal_mask.int())
print(f"\nInterpretation:")
print(f"Position 0 can attend to: [0]")
print(f"Position 1 can attend to: [0, 1]")
print(f"Position 2 can attend to: [0, 1, 2]")
print(f"Position 3 can attend to: [0, 1, 2, 3]")
print(f"...and so on")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(causal_mask.int(), cmap='Blues')
plt.colorbar(label='Can Attend (1) or Cannot (0)')
plt.title(f'Causal Mask (Seq Len = {seq_len})')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.xticks(range(seq_len))
plt.yticks(range(seq_len))
plt.tight_layout()
plt.show()

## Part 4: Transformer Decoder Architecture

Understanding different decoder architectures and how they use attention.

### 4.1 Encoder-Decoder Architecture (Seq2Seq)

**Components:**
1. Encoder: Processes input sequence using self-attention
2. Decoder: Generates output using masked self-attention + cross-attention

**Process:**
- Encoder reads entire input and produces context
- Decoder generates output token by token
- Decoder attends to encoder outputs via cross-attention

**Advantages:**
- Suitable for variable input/output lengths
- Encoder can attend bidirectionally
- Good for translation, summarization

**Disadvantages:**
- Separate encoder and decoder
- More complex training

**Best for:** Machine translation, summarization, question answering

In [ ]:
print("\n" + "=" * 70)
print("ENCODER-DECODER ARCHITECTURE")
print("=" * 70)

print("""
Architecture:

ENcoder:
  Input: "Hello world"
  ↓
  Token Embedding + Positional Encoding
  ↓
  Stack of Encoder Layers (Self-Attention + FFN)
  ↓
  Context: [context_vector_1, context_vector_2]

Decoder:
  Decoder Input: "Hola mundo" (with START token)
  ↓
  Token Embedding + Positional Encoding
  ↓
  Stack of Decoder Layers:
    - Masked Self-Attention (attend to previous tokens only)
    - Cross-Attention (attend to encoder outputs)
    - Feed-Forward Network
  ↓
  Output: Logits for next token
  ↓
  Apply decoding strategy (Greedy, Beam Search, etc.)
  ↓
  Next Token: "mundo"

""")

print("TRAINING vs INFERENCE")
print("-" * 50)
print("Training: Teacher Forcing")
print("  - All decoder inputs provided at once")
print("  - Parallel decoding across sequence")
print("  - Fast but assumes perfect input")
print()
print("Inference: Autoregressive")
print("  - Generate one token at a time")
print("  - Use previous output as next input")
print("  - Slower but can fix errors")

### 4.2 Decoder-Only Architecture (Causal Language Model)

**Components:**
- Single stack of decoder layers with causal masking
- No separate encoder
- All attention is self-attention with masking

**Process:**
- Input: context tokens
- Self-attention with causal mask
- Predict next token
- Repeat

**Advantages:**
- Simple architecture
- Efficient for generation
- Great for language modeling
- Used by GPT, LLaMA, etc.

**Disadvantages:**
- Limited to next-token prediction during training
- Cannot attend to future tokens

**Best for:** Language modeling, text generation

In [ ]:
print("\n" + "=" * 70)
print("DECODER-ONLY ARCHITECTURE")
print("=" * 70)

print("""
Architecture:

Input: "The quick brown fox"
↓
Token Embedding + Positional Encoding
↓
Stack of Decoder Layers:
  - Masked Self-Attention (causal mask)
  - Feed-Forward Network
  (Repeat for N layers)
↓
Output Projection → Logits
↓
Next Token: "jumps"

Example Generation Process:

Context:   ["The"]             → Next: "quick"
Context:   ["The", "quick"]     → Next: "brown"
Context:   ["The", "quick", "brown"] → Next: "fox"

Key Difference from Encoder-Decoder:
- No separate encoder
- No cross-attention
- ONLY self-attention with causal mask
- Position i can only see positions 0 to i-1

Used by: GPT-2, GPT-3, GPT-4, LLaMA, Falcon, etc.
""")

print("\nComparison Table:")
print("-" * 70)
print(f"{'Property':<30} {'Encoder-Decoder':<20} {'Decoder-Only':<20}")
print("-" * 70)
print(f"{'Architecture':<30} {'Separate enc/dec':<20} {'Single stack':<20}")
print(f"{'Attention Types':<30} {'Self + Cross':<20} {'Self (causal)':<20}")
print(f"{'Bidirectional Encoding':<30} {'Yes':<20} {'No':<20}")
print(f"{'Parameters':<30} {'More':<20} {'Less':<20}")
print(f"{'Training Speed':<30} {'Slower':<20} {'Faster':<20}")
print(f"{'Inference Speed':<30} {'Slower':<20} {'Faster':<20}")
print(f"{'Best for':<30} {'Translation':<20} {'Generation':<20}")

## Part 5: Decoding Strategies

Different methods to convert model outputs to sequences during inference.

### 5.1 Greedy Decoding

**Process:**
1. Generate next token: argmax(logits)
2. Add to sequence
3. Repeat until [EOS] token

**Advantages:**
- Fastest method
- Simple to implement
- Deterministic

**Disadvantages:**
- Often produces suboptimal results
- No backtracking
- Can get stuck in loops (repetition)

**Best for:** Fast inference, real-time applications

In [ ]:
def greedy_decode(logits: torch.Tensor, eos_token: int) -> list:
    """
    Greedy decoding: select token with highest probability at each step.
    
    Args:
        logits: Model output logits
        eos_token: End of sequence token ID
    
    Returns:
        Decoded token sequence
    """
    predicted_tokens = []
    current_logits = logits
    
    while True:
        # Get token with highest probability
        next_token = torch.argmax(current_logits[-1, -1, :]).item()
        predicted_tokens.append(next_token)
        
        if next_token == eos_token or len(predicted_tokens) > 100:
            break
        
        # In real scenario, we would:
        # 1. Add next_token to sequence
        # 2. Get new logits from model
        # 3. Continue
    
    return predicted_tokens

print("\n" + "=" * 70)
print("GREEDY DECODING")
print("=" * 70)

print("""
Algorithm:

1. Get logits from model for current sequence
2. Select token with highest probability: argmax(logits)
3. Add token to sequence
4. Repeat steps 1-3 until:
   - [EOS] token generated, OR
   - Max length reached

Example:

Input: "What is artificial"
↓
Logits distribution for next word:
  intelligence: 0.45
  learning: 0.30
  systems: 0.15
  ...
↓
Greedy selects: "intelligence" (highest probability)
↓
Output: "What is artificial intelligence"

""")

print("PROS & CONS:")
print("-" * 50)
print("✓ Fastest decoding method")
print("✓ Simplest to implement")
print("✗ Often produces suboptimal text")
print("✗ Prone to repetition loops")
print("✗ No chance to correct mistakes")

print("\nComplexity: O(seq_len) with single forward pass per token")

### 5.2 Beam Search

**Process:**
1. Keep top-K sequences with highest cumulative probability
2. Expand each sequence by one token
3. Prune to keep top-K again
4. Repeat until all sequences reach [EOS]

**Advantages:**
- Better quality than greedy
- Explores multiple hypotheses
- Finds higher probability sequences

**Disadvantages:**
- Slower than greedy
- Can be repetitive (needs length penalty)
- Requires storing K sequences

**Best for:** Quality-critical applications, translation

In [ ]:
class BeamSearchDecoder:
    """Beam Search decoding with length penalty"""
    
    def __init__(self, beam_width: int = 3, length_penalty: float = 1.0):
        self.beam_width = beam_width
        self.length_penalty = length_penalty
    
    def decode(self, initial_logits: torch.Tensor, eos_token: int, max_length: int = 20) -> list:
        """
        Args:
            initial_logits: First token logits of shape (vocab_size,)
            eos_token: End of sequence token
            max_length: Maximum generation length
        
        Returns:
            List of top-k decoded sequences
        """
        batch_size = 1
        vocab_size = initial_logits.size(-1)
        
        # Initialize: top beam_width tokens from initial logits
        log_probs, beam_indices = torch.topk(
            torch.log_softmax(initial_logits, dim=-1),
            self.beam_width
        )
        
        # Store sequences
        sequences = [[idx.item()] for idx in beam_indices]
        scores = log_probs.tolist()
        
        # Iteratively expand beams
        for step in range(1, max_length):
            new_sequences = []
            new_scores = []
            
            for seq_idx, (sequence, score) in enumerate(zip(sequences, scores)):
                if sequence[-1] == eos_token:
                    # Sequence finished, keep as is
                    new_sequences.append(sequence)
                    new_scores.append(score)
                    continue
                
                # Simulate getting next logits
                # In real implementation: model(sequence)
                next_logits = torch.randn(vocab_size)
                next_log_probs = torch.log_softmax(next_logits, dim=-1)
                
                # Apply length penalty
                adjusted_log_probs = next_log_probs / (step ** self.length_penalty)
                
                # Get top candidates
                top_log_probs, top_indices = torch.topk(adjusted_log_probs, 1)
                
                for next_token_idx in range(1):
                    new_score = score + top_log_probs[next_token_idx].item()
                    new_seq = sequence + [top_indices[next_token_idx].item()]
                    new_sequences.append(new_seq)
                    new_scores.append(new_score)
            
            # Keep top beam_width sequences
            sorted_indices = sorted(
                range(len(new_scores)), 
                key=lambda i: new_scores[i], 
                reverse=True
            )[:self.beam_width]
            
            sequences = [new_sequences[i] for i in sorted_indices]
            scores = [new_scores[i] for i in sorted_indices]
        
        return sequences, scores

print("\n" + "=" * 70)
print("BEAM SEARCH")
print("=" * 70)

print("""
Algorithm (Beam Width = 3):

Step 1: Get initial probabilities
  intelligence: 0.45  ← Select
  learning: 0.30      ← Select
  systems: 0.15       ← Select
  
  Active beams: [intelligence], [learning], [systems]

Step 2: Expand each beam
  [intelligence]
    ├─ [intelligence, model]: 0.35
    ├─ [intelligence, is]: 0.25
    └─ [intelligence, system]: 0.20
  
  [learning]
    ├─ [learning, is]: 0.30
    ├─ [learning, model]: 0.20
    └─ [learning, system]: 0.15
  
  [systems]
    ├─ [systems, are]: 0.18
    ├─ [systems, is]: 0.10
    └─ [systems, model]: 0.05

Step 3: Keep top 3 by cumulative probability
  [intelligence, model]: 0.45 × 0.35 = 0.158
  [intelligence, is]: 0.45 × 0.25 = 0.113
  [learning, is]: 0.30 × 0.30 = 0.090

Repeat until [EOS] token or max length

""")

print("PROS & CONS:")
print("-" * 50)
print("✓ Better quality than greedy")
print("✓ Explores multiple hypotheses")
print("✓ Finds higher probability sequences")
print("✗ Slower (K times slower than greedy)")
print("✗ Can still be repetitive without length penalty")
print("✗ More memory usage")

print(f"\nComplexity: O(K × seq_len × vocab_size)")
print(f"where K = beam_width")

### 5.3 Top-K Sampling

**Process:**
1. Keep only top-K tokens by probability
2. Renormalize probabilities
3. Sample from distribution
4. Add to sequence
5. Repeat

**Advantages:**
- More diverse output
- Avoids low-probability tokens
- Better than greedy for natural text

**Disadvantages:**
- Can be non-deterministic
- May still sample unlikely tokens
- K is a hyperparameter to tune

**Best for:** Open-ended text generation

In [ ]:
def top_k_sampling(logits: torch.Tensor, k: int = 10, temperature: float = 1.0) -> int:
    """
    Top-K sampling for token selection.
    
    Args:
        logits: Model output logits of shape (vocab_size,)
        k: Keep only top-k tokens
        temperature: Softmax temperature (>1 makes more uniform, <1 makes more peaked)
    
    Returns:
        Sampled token ID
    """
    # Apply temperature
    scaled_logits = logits / temperature
    
    # Get probabilities
    probs = torch.softmax(scaled_logits, dim=-1)
    
    # Get top-k
    top_k_probs, top_k_indices = torch.topk(probs, k)
    
    # Renormalize
    top_k_probs = top_k_probs / top_k_probs.sum()
    
    # Sample
    sampled_idx = torch.multinomial(top_k_probs, 1)
    sampled_token = top_k_indices[sampled_idx].item()
    
    return sampled_token, top_k_probs[sampled_idx].item()

print("\n" + "=" * 70)
print("TOP-K SAMPLING")
print("=" * 70)

print("""
Algorithm:

1. Get next token logits from model
2. Convert to probabilities with softmax
3. Select top-K tokens by probability
4. Renormalize probabilities (must sum to 1)
5. Sample from renormalized distribution
6. Add sampled token to sequence
7. Repeat

Example (K=3, vocabulary size=50000):

All token probabilities:
  the: 0.15
  is: 0.12
  cat: 0.10
  dog: 0.08
  .... (49996 more tokens with very low probabilities)
  
Step 3: Select top-3
  the: 0.15 ← 
  is: 0.12  ← 
  cat: 0.10 ← 
  (Rest discarded: probability ≈ 0)

Step 4: Renormalize
  the: 0.15 / 0.37 = 0.405
  is: 0.12 / 0.37 = 0.324
  cat: 0.10 / 0.37 = 0.270

Step 5: Sample from [the, is, cat]
  Random roll... → Select "is"

""")

print("PROS & CONS:")
print("-" * 50)
print("✓ Diverse outputs")
print("✓ Avoids very low probability tokens")
print("✓ Faster than beam search")
print("✗ Non-deterministic")
print("✗ K is a hyperparameter to tune")
print("✗ Might miss rare but good words")

print("\nWhen to use different K values:")
print("-" * 50)
print("K=1: Equivalent to greedy (deterministic)")
print("K=10: Conservative, pick from 10 best options")
print("K=50: More diverse, pick from 50 best options")
print("K=full vocab: Like standard sampling (can be bad)")

### 5.4 Top-P (Nucleus) Sampling

**Process:**
1. Sort tokens by probability (descending)
2. Keep tokens until cumulative probability ≥ P
3. Renormalize and sample

**Advantages:**
- Adapts to distribution (no fixed K)
- Keeps "nucleus" of distribution
- Very natural text generation

**Disadvantages:**
- Slightly more complex
- P is a hyperparameter

**Best for:** High-quality text generation (GPT-3, ChatGPT)

In [ ]:
def top_p_sampling(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> int:
    """
    Top-P (Nucleus) sampling for token selection.
    
    Args:
        logits: Model output logits of shape (vocab_size,)
        p: Cumulative probability threshold
        temperature: Softmax temperature
    
    Returns:
        Sampled token ID
    """
    # Apply temperature
    scaled_logits = logits / temperature
    
    # Get probabilities
    probs = torch.softmax(scaled_logits, dim=-1)
    
    # Sort by probability
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    
    # Compute cumulative probabilities
    cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # Find cutoff: keep tokens where cumsum <= p
    sorted_indices_to_remove = cumsum_probs > p
    sorted_indices_to_remove[0] = False  # Keep at least one token
    
    # Remove low probability tokens
    sorted_probs[sorted_indices_to_remove] = 0
    
    # Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum()
    
    # Sample
    sampled_idx = torch.multinomial(sorted_probs, 1)
    sampled_token = sorted_indices[sampled_idx].item()
    
    return sampled_token

print("\n" + "=" * 70)
print("TOP-P (NUCLEUS) SAMPLING")
print("=" * 70)

print("""
Algorithm:

1. Get next token probabilities from model
2. Sort tokens by probability (highest first)
3. Accumulate probabilities until sum ≥ P
4. Keep those tokens (discard rest)
5. Renormalize and sample

Example (P=0.9):

Sorted probabilities:
  Token A: 0.50  ← Cumsum: 0.50
  Token B: 0.30  ← Cumsum: 0.80
  Token C: 0.10  ← Cumsum: 0.90  ← Reaches threshold
  Token D: 0.05  ← Cumsum: 0.95  ← REMOVE (> 0.9)
  Token E: 0.03  ← Cumsum: 0.98  ← REMOVE
  Token F: 0.02  ← Cumsum: 1.00  ← REMOVE

Nucleus (kept tokens): [A, B, C]

Renormalize:
  Token A: 0.50 / 0.90 = 0.556
  Token B: 0.30 / 0.90 = 0.333
  Token C: 0.10 / 0.90 = 0.111

Sample from nucleus: 
  P(A) = 55.6%, P(B) = 33.3%, P(C) = 11.1%

""")

print("PROS & CONS:")
print("-" * 50)
print("✓ Adapts to distribution (P is universal)")
print("✓ Produces natural, diverse text")
print("✓ Used by GPT-3, ChatGPT, and other SoTA models")
print("✗ Slightly more complex than Top-K")
print("✗ P is still a hyperparameter")

print("\nUsual settings:")
print("-" * 50)
print("P=0.9: Standard, balances diversity and coherence")
print("P=0.95: More conservative, less diversity")
print("P=1.0: All tokens (but sorted by prob)")
print()
print("Recommended for most text generation tasks!")

### 5.5 Temperature-Based Sampling

**Process:**
- Divide logits by temperature before softmax
- Temperature < 1: Makes distribution more peaked (deterministic)
- Temperature > 1: Makes distribution more uniform (diverse)

**Advantages:**
- Simple to implement
- Effective way to control randomness
- Often combined with Top-K or Top-P

**Disadvantages:**
- Linear scaling might not be ideal
- Hyperparameter to tune

**Best for:** Fine-tuning generation diversity

In [ ]:
def temperature_sampling(logits: torch.Tensor, temperature: float = 1.0) -> int:
    """
    Standard sampling with temperature control.
    
    Args:
        logits: Model output logits
        temperature: Sampling temperature
               < 1.0: More deterministic
               = 1.0: Standard
               > 1.0: More diverse
    
    Returns:
        Sampled token ID
    """
    scaled_logits = logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    sampled_token = torch.multinomial(probs, 1).item()
    return sampled_token

print("\n" + "=" * 70)
print("TEMPERATURE-BASED SAMPLING")
print("=" * 70)

print("""
Concept:

logits: [1.0, 2.0, 3.0, 0.5, 0.2]

Temperature = 0.5 (Cold - Deterministic):
  scaled: [2.0, 4.0, 6.0, 1.0, 0.4]
  softmax → [0.001, 0.02, 0.97, 0.005, 0.004]  ← Peaks heavily on highest
  
Temperature = 1.0 (Default):
  scaled: [1.0, 2.0, 3.0, 0.5, 0.2]
  softmax → [0.006, 0.015, 0.041, 0.003, 0.002]  ← Balanced
  
Temperature = 2.0 (Hot - Diverse):
  scaled: [0.5, 1.0, 1.5, 0.25, 0.1]
  softmax → [0.3, 0.4, 0.5, 0.25, 0.2]  ← Spread out

""")

# Visualize effect of temperature
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.2])
temperatures = [0.5, 1.0, 1.5, 2.0]

plt.figure(figsize=(12, 4))
for i, temp in enumerate(temperatures, 1):
    probs = torch.softmax(logits / temp, dim=-1)
    plt.subplot(1, 4, i)
    plt.bar(range(5), probs.detach().numpy())
    plt.title(f'Temperature={temp}')
    plt.ylabel('Probability')
    plt.ylim([0, 1])
    plt.xticks(range(5))

plt.suptitle('Effect of Temperature on Probability Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print("\nTemperature Guidelines:")
print("-" * 50)
print("0.0-0.5: Very deterministic, best for factual QA")
print("0.5-1.0: Balanced, good for most tasks")
print("1.0: Default, neutral randomness")
print("1.0-1.5: Creative, good for story generation")
print("1.5-2.0: Very creative, might be incoherent")
print(">2.0: Too random, usually produces gibberish")

## Part 6: Comprehensive Comparison and Best Practices

Comparing all methods and their practical applications.

In [ ]:
import pandas as pd

print("\n" + "=" * 100)
print("DECODING METHODS COMPARISON")
print("=" * 100)

comparison_data = {
    'Method': ['Greedy', 'Beam Search', 'Top-K', 'Top-P', 'Temperature'],
    'Speed': ['Fastest', 'Slow', 'Fast', 'Fast', 'Fast'],
    'Quality': ['Low', 'High', 'Medium-High', 'High', 'Medium'],
    'Diversity': ['None', 'Medium', 'High', 'High', 'High'],
    'Repetition': ['High', 'Medium-High', 'Low', 'Low', 'Medium'],
    'Parameters': [0, 1, 1, 1, 1],
    'Use Case': ['Real-time', 'Translation', 'Open-ended', 'LLMs', 'Diversity Control'],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

print("\n" + "-" * 100)
print("POSITIONAL ENCODING COMPARISON")
print("-" * 100)

pe_data = {
    'Method': ['Sinusoidal', 'Learned', 'Relative', 'ALiBi', 'RoPE'],
    'Learnable': ['No', 'Yes', 'Yes', 'Yes', 'No'],
    'Parameters': ['0', 'max_len × d_model', 'Low', 'Low', '0'],
    'Extrapolation': ['Good', 'Poor', 'Good', 'Excellent', 'Excellent'],
    'Memory': ['O(1)', 'O(max_len)', 'O(max_rel_dist)', 'O(num_heads)', 'O(1)'],
    'Complexity': ['Low', 'Low', 'Medium', 'Low', 'Medium'],
    'Popular Models': ['BERT', 'Early work', 'T5', 'MPT', 'LLaMA, PaLM, GPT-NeoX'],
}

df_pe = pd.DataFrame(pe_data)
print(df_pe.to_string(index=False))

print("\n" + "-" * 100)
print("ATTENTION TYPES COMPARISON")
print("-" * 100)

att_data = {
    'Attention Type': ['Self-Attention (Encoder)', 'Self-Attention (Causal)', 'Cross-Attention', 'Multi-Head'],
    'Q Source': ['Input', 'Input', 'Decoder', 'Input'],
    'K,V Source': ['Input', 'Input', 'Encoder', 'Input'],
    'Masking': ['Padding only', 'Causal + Padding', 'Padding only', 'Varies'],
    'Direction': ['Bidirectional', 'Unidirectional', 'N/A', 'Varies'],
    'Use': ['Encoder layers', 'Decoder (LM)', 'Seq2seq decoder', 'All transformers'],
}

df_att = pd.DataFrame(att_data)
print(df_att.to_string(index=False))

### Choosing the Right Configuration

A practical guide for selecting encoding and decoding methods for different tasks.

In [ ]:
print("\n" + "=" * 100)
print("TASK-SPECIFIC RECOMMENDATIONS")
print("=" * 100)

task_recommendations = {
    'Task': [
        'Machine Translation',
        'Question Answering',
        'Text Summarization',
        'Language Modeling',
        'Dialogue / Chatbot',
        'Code Generation',
        'Sentiment Analysis',
        'Named Entity Recognition',
    ],
    'Encoder': [
        'Self-Attention (Bidirectional)',
        'Self-Attention (Bidirectional)',
        'Self-Attention (Bidirectional)',
        'Causal Self-Attention',
        'Self-Attention (Bidirectional)',
        'Causal Self-Attention',
        'Self-Attention (Bidirectional)',
        'Self-Attention (Bidirectional)',
    ],
    'Positional Encoding': [
        'RoPE or Sinusoidal',
        'Sinusoidal or ALiBi',
        'RoPE or Sinusoidal',
        'RoPE',
        'RoPE or Sinusoidal',
        'RoPE',
        'Sinusoidal',
        'Sinusoidal',
    ],
    'Decoder/Generation': [
        'Beam Search (K=4-8)',
        'Greedy or Beam',
        'Beam Search (K=5-10)',
        'Top-P or Top-K',
        'Top-P (p=0.9)',
        'Top-P or Greedy',
        'N/A (Classification)',
        'N/A (Tagging)',
    ],
    'Architecture': [
        'Encoder-Decoder',
        'Encoder-Decoder',
        'Encoder-Decoder',
        'Decoder-Only',
        'Decoder-Only',
        'Decoder-Only',
        'Encoder-Only',
        'Encoder-Only',
    ],
}

df_tasks = pd.DataFrame(task_recommendations)
for col in df_tasks.columns:
    print(f"\n{col}:")
    for i, val in enumerate(df_tasks[col].values):
        if i == 0:
            continue
        print(f"  {df_tasks['Task'].values[i]}: {val}")

### Real-World Configuration Examples

In [ ]:
print("\n" + "=" * 100)
print("REAL-WORLD CONFIGURATION EXAMPLES")
print("=" * 100)

print("""
╔════════════════════════════════════════════════════════════════════════════════════╗
║ EXAMPLE 1: Machine Translation (like Google Translate)                             ║
╚════════════════════════════════════════════════════════════════════════════════════╝

Architecture: Encoder-Decoder (Transformer Base)

Encoder:
  - Input Embedding: vocab_size=32000 → d_model=512
  - Positional Encoding: RoPE
  - Attention: Multi-Head (8 heads)
  - Layers: 6
  - Feed-Forward: d_ff=2048

Decoder:
  - Input Embedding: vocab_size=32000 → d_model=512
  - Positional Encoding: RoPE
  - Masked Self-Attention: Multi-Head (8 heads)
  - Cross-Attention: To encoder outputs
  - Layers: 6
  - Feed-Forward: d_ff=2048

Generation:
  - Decoding Method: Beam Search (K=5)
  - Temperature: 0.8 (slightly creative)
  - Length Penalty: 0.6 (prefer longer sequences)
  - Max Length: 256 tokens
  - EOS Token: </s>

Rationale:
  ✓ Encoder processes full source sentence
  ✓ Cross-attention aligns with source
  ✓ Beam search ensures high quality
  ✓ RoPE handles variable length

╔════════════════════════════════════════════════════════════════════════════════════╗
║ EXAMPLE 2: Chatbot (like ChatGPT)                                                 ║
╚════════════════════════════════════════════════════════════════════════════════════╝

Architecture: Decoder-Only (Causal Language Model)

Model:
  - Input Embedding: vocab_size=100000 → d_model=768
  - Positional Encoding: RoPE
  - Attention: Multi-Head (12 heads)
  - Layers: 24
  - Feed-Forward: d_ff=3072
  - Context Window: 2048 tokens

Generation:
  - Decoding Method: Top-P Sampling (p=0.9)
  - Temperature: 0.7-1.0 (diverse but coherent)
  - Max Length: 1024 tokens
  - No Beam Search (too slow for dialogue)
  - Stops at: </s> or max length

Rationale:
  ✓ Simple decoder-only architecture (fast)
  ✓ Top-P sampling for natural dialogue
  ✓ High temperature for creativity
  ✓ RoPE allows long context
  ✓ Fast inference for real-time chat

╔════════════════════════════════════════════════════════════════════════════════════╗
║ EXAMPLE 3: Question Answering (like SQuAD)                                        ║
╚════════════════════════════════════════════════════════════════════════════════════╝

Architecture: Encoder-Only (like BERT)

Model:
  - Input Embedding: vocab_size=30522 → d_model=768
  - Token Type Embedding: 2 segments
  - Positional Embedding: Learned (max_seq=512)
  - Attention: Multi-Head (12 heads)
  - Layers: 12
  - Feed-Forward: d_ff=3072

Inference:
  - No decoding needed (span extraction)
  - Just find start and end token positions
  - No sampling or beam search
  - Outputs: start_logits, end_logits

Rationale:
  ✓ Bidirectional attention (full context)
  ✓ Fast inference (single pass)
  ✓ Learned positional encoding works for fixed length
  ✓ No need for decoding (classification task)

╔════════════════════════════════════════════════════════════════════════════════════╗
║ EXAMPLE 4: Code Generation (like GitHub Copilot)                                  ║
╚════════════════════════════════════════════════════════════════════════════════════╝

Architecture: Decoder-Only with special tokens

Model:
  - Input Embedding: vocab_size=50000 → d_model=1024
  - Positional Encoding: RoPE (supports up to 2048 context)
  - Attention: Multi-Head (16 heads)
  - Layers: 36
  - Feed-Forward: d_ff=4096
  - Special Tokens: <|py|>, <|js|>, etc.

Generation:
  - Decoding Method: Top-P Sampling (p=0.95) or Greedy
  - Temperature: 0.5-0.8 (correctness over creativity)
  - Max Tokens: 256-512
  - Deterministic Sampling: For reproducibility

Rationale:
  ✓ Lower temperature for correct syntax
  ✓ Top-P for some diversity (multiple valid solutions)
  ✓ Special tokens for language indication
  ✓ Greedy acceptable if temperature is low enough
  ✓ Long context for seeing full function/file
""")

### Best Practices and Common Pitfalls

In [ ]:
print("\n" + "=" * 100)
print("BEST PRACTICES & COMMON PITFALLS")
print("=" * 100)

print("""
╔════════════════════════════════════════════════════════════════════════════════════╗
║ BEST PRACTICES                                                                     ║
╚════════════════════════════════════════════════════════════════════════════════════╝

1. POSITIONAL ENCODING
   ✓ Use RoPE for long sequences (supports extrapolation)
   ✓ Use Sinusoidal for standard tasks (proven, no params)
   ✓ Use Learned PE only if sequence length is fixed
   ✓ Validate on sequences 2x longer than training

2. ATTENTION MECHANISMS
   ✓ Use Multi-Head Attention (8-16 heads typical)
   ✓ Separate encoder (bidirectional) from decoder (causal)
   ✓ Use causal masking in decoder to prevent peeking
   ✓ Always apply dropout to attention weights

3. DECODING STRATEGIES
   ✓ Use Beam Search for quality-critical tasks (translation)
   ✓ Use Top-P for open-ended generation (creative writing)
   ✓ Use Greedy with low temperature for factual tasks
   ✓ Combine methods: Top-P + Temperature is standard

4. TEMPERATURE TUNING
   ✓ Start at temperature=0.7 for most tasks
   ✓ Lower (0.3-0.5) for factual/code generation
   ✓ Higher (1.0-1.2) for creative writing
   ✓ Always experiment with your specific task

5. MEMORY & SPEED
   ✓ Cache attention weights during generation
   ✓ Use KV-caching to avoid recomputation
   ✓ Batch sequences when possible
   ✓ Use quantization for inference (FP16, INT8)

╔════════════════════════════════════════════════════════════════════════════════════╗
║ COMMON PITFALLS                                                                    ║
╚════════════════════════════════════════════════════════════════════════════════════╝

1. POSITION ENCODING MISTAKES
   ✗ Using learned PE with variable length inputs
   ✗ Not scaling embeddings by √d_model
   ✗ Using same PE for both encoder and decoder
   ✗ Assuming sinusoidal PE works for very long sequences
   
   Fix: Match PE to your use case, test extrapolation

2. ATTENTION MASK MISTAKES
   ✗ Forgetting padding mask in attention
   ✗ Not applying causal mask in decoder
   ✗ Using 0 instead of -inf for masked positions
   ✗ Masking mistakes during cross-attention
   
   Fix: Carefully check mask dimensions and application

3. DECODING MISTAKES
   ✗ Using greedy decoding for important tasks
   ✗ Setting temperature too high (incoherent output)
   ✗ Forgetting EOS token (infinite loops)
   ✗ Not normalizing beam search scores by length
   
   Fix: Use appropriate decoding, test thoroughly

4. SEQUENCE LENGTH ISSUES
   ✗ Training on short sequences, testing on long
   ✗ Not handling variable length in batches
   ✗ OOM errors during inference with long sequences
   ✗ Using insufficient context for complex tasks
   
   Fix: Train on expected length distribution, use sliding window

5. GENERATION QUALITY ISSUES
   ✗ Repetitive text (need length penalty in beam search)
   ✗ Inconsistent outputs (temperature too high)
   ✗ Low diversity (temperature too low)
   ✗ Poor long-term coherence (not enough context)
   
   Fix: Tune temperature, use Top-P, increase context window

╔════════════════════════════════════════════════════════════════════════════════════╗
║ DEBUGGING CHECKLIST                                                                ║
╚════════════════════════════════════════════════════════════════════════════════════╝

□ Check attention mask shapes match (batch, heads, seq_len, seq_len)
□ Verify positional encoding ranges work for your sequence lengths
□ Test on longer sequences than training data
□ Check for NaN/Inf in attention weights
□ Validate that attention sums to 1 along the key dimension
□ Test decoding independently from model
□ Compare outputs with reference implementations
□ Profile memory usage during inference
□ Monitor temperature and top-p parameter effects
□ Check for repetition in generated text
""")

## Summary Table: Quick Reference

In [ ]:
print("\n" + "=" * 100)
print("QUICK REFERENCE: WHEN TO USE WHAT")
print("=" * 100)

print("""
┌─────────────────┬──────────────────────┬──────────────┬────────────────────────┐
│ COMPONENT       │ OPTION 1             │ OPTION 2     │ WHEN TO USE OPTION 1   │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Positional      │ RoPE                 │ Sinusoidal   │ Long sequences (RoPE)  │
│ Encoding        │                      │              │ Standard (Sinusoidal)  │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Attention Type  │ Self-Attention       │ Cross-Att.   │ Within sequence        │
│                 │                      │              │ Between sequences      │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Masking         │ Causal Mask          │ Padding      │ Decoder (next token)   │
│                 │                      │              │ Padding (both)         │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Generation      │ Beam Search          │ Top-P        │ Translation (Beam)     │
│ Strategy        │                      │              │ Chat/Creative (Top-P)  │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Temperature     │ 0.3-0.7              │ 0.8-1.2      │ Factual (low temp)     │
│                 │                      │              │ Creative (high temp)   │
├─────────────────┼──────────────────────┼──────────────┼────────────────────────┤
│ Architecture    │ Encoder-Decoder      │ Decoder-Only │ Seq2Seq (Enc-Dec)      │
│                 │                      │              │ Language Model (Dec)   │
└─────────────────┴──────────────────────┴──────────────┴────────────────────────┘

QUICK DECISION TREE:

Is output variable length?
  ├─ YES: Need encoding/decoding?
  │   ├─ YES: Encoder-Decoder + Beam Search
  │   └─ NO: Decoder-Only + Top-P Sampling
  └─ NO: Fixed classification/tagging?
      └─ YES: Encoder-Only + Softmax

Is sequence very long (>1000 tokens)?
  ├─ YES: Use RoPE or ALiBi
  └─ NO: Use Sinusoidal or Learned

Need high quality (e.g., translation)?
  ├─ YES: Use Beam Search
  └─ NO: Use Top-P or Top-K for speed

Need deterministic output?
  ├─ YES: Use Greedy or Temperature=0
  └─ NO: Use Top-P or Temperature=0.7-1.0
""")

print("\n" + "=" * 100)
print("END OF COMPREHENSIVE GUIDE")
print("=" * 100)

## Additional Resources and References

Key papers and implementations:

1. **Original Transformer**: "Attention Is All You Need" (Vaswani et al., 2017)
2. **Relative PE**: "Relative Position Representations for Structural Relation Extraction" (Shaw et al., 2018)
3. **ALiBi**: "Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation" (Press et al., 2022)
4. **RoPE**: "RoFormer: Enhanced Transformer with Rotary Position Embedding" (Su et al., 2022)
5. **Top-P Sampling**: "The Curious Case of Neural Text Degeneration" (Holtzman et al., 2020)
6. **BERT**: "BERT: Pre-training of Deep Bidirectional Transformers" (Devlin et al., 2019)
7. **GPT Series**: "Language Models are Unsupervised Multitask Learners" (Radford et al., 2019)

## Code Implementation Notes

For production implementations, consider:
- Use established libraries: HuggingFace Transformers, PyTorch, TensorFlow
- Implement KV-caching for efficient inference
- Use flash attention for faster computation
- Consider quantization for deployment
- Test extensively on your specific domain